# 第二阶段：特征预处理与有效性验证

本 Notebook 基于特征工程产物完成建模前预处理和特征有效性初步验证，包括数值特征标准化、高维类别特征 Target Encoding、方差阈值筛选、相关性分析、互信息分析和特征重要性预评估。

注意：Target Encoding 和标准化均只在训练集上拟合，再应用到测试集，避免全量数据拟合造成数据泄露。

## 1. 导入工具库与路径配置

In [1]:
# 配置特征预处理所需路径，并导入编码、标准化和特征评估工具。
from pathlib import Path
import re

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import VarianceThreshold, mutual_info_classif
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler


PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

FEATURE_INPUT_PATH = PROJECT_ROOT / "data" / "processed" / "hotel_booking_features.parquet"
MODELING_DATASET_PATH = PROJECT_ROOT / "data" / "processed" / "modeling_dataset.parquet"
TRAIN_DATASET_PATH = PROJECT_ROOT / "data" / "processed" / "modeling_dataset_train.parquet"
VALIDATION_DATASET_PATH = PROJECT_ROOT / "data" / "processed" / "modeling_dataset_validation.parquet"
TEST_DATASET_PATH = PROJECT_ROOT / "data" / "processed" / "modeling_dataset_test.parquet"
PREPROCESSING_SUMMARY_PATH = PROJECT_ROOT / "reports" / "feature_preprocessing_summary.csv"
FEATURE_SELECTION_REPORT_PATH = PROJECT_ROOT / "reports" / "feature_selection_report.csv"
FEATURE_IMPORTANCE_REPORT_PATH = PROJECT_ROOT / "reports" / "feature_importance_preview.csv"

TARGET_COLUMN = "is_canceled"
RANDOM_STATE = 42

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)

FEATURE_INPUT_PATH

WindowsPath('D:/携程实习文件/hotel-booking-analysis/data/processed/hotel_booking_features.parquet')

## 2. 读取特征工程数据

In [2]:
# 读取特征工程结果，作为后续建模预处理的输入。
if not FEATURE_INPUT_PATH.exists():
    raise FileNotFoundError(f"特征工程数据不存在：{FEATURE_INPUT_PATH}")

df = pd.read_parquet(FEATURE_INPUT_PATH)
print(f"特征工程数据规模：{df.shape[0]:,} 行，{df.shape[1]:,} 列")
print(f"目标变量均值：{df[TARGET_COLUMN].mean():.2%}")
df.head()

特征工程数据规模：110,628 行，86 列
目标变量均值：37.08%


,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,children,babies,meal,country,market_segment,distribution_channel,is_repeated_guest,previous_cancellations,previous_bookings_not_canceled,reserved_room_type,assigned_room_type,booking_changes,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date,city,total_stays,arrival_date,hotel_type,arrival_month_num,arrival_month,arrival_quarter,arrival_quarter_label,arrival_day_of_week,is_weekend,is_holiday_proxy,holiday_proxy,is_near_holiday_proxy,season_type,is_peak_season,is_low_season,total_guests,has_children_or_babies,room_type_matched,has_booking_changes,has_special_requests,has_car_parking_request,lead_time_group,special_request_group,customer_group_key,estimated_revenue,group_recency_days,group_booking_count,group_avg_revenue,rfm_value_score,rfm_segment,is_high_value_customer_group,customer_group_hist_booking_count,customer_group_hist_cancel_rate,country_hist_booking_count,country_hist_cancel_rate,agent_hist_booking_count,agent_hist_cancel_rate,hotel_hist_booking_count,hotel_hist_cancel_rate,is_high_risk_country,is_high_risk_agent,hotel_hist_avg_adr,city_hist_avg_adr,customer_group_hist_avg_adr,price_to_hotel_hist_avg,price_to_city_hist_avg,price_to_customer_group_hist_avg,price_volatility_index,price_sensitivity_score,hotel_daily_booking_count,estimated_occupancy_proxy,demand_pressure_score,cancellation_risk_score
0,Resort Hotel - Chandigarh,0,342,2024,July,30,27,0,0,2,0.0,0,BB,PRT,Direct,Direct,0,0,0,C,C,3,No Deposit,0.0,0.0,0,Transient,0.0,0,0,Check-Out,2024-07-27 22:16:40.916332324,Chandigarh,0,2024-07-27,Resort Hotel,7,2024-07,3,2024Q3,5,1,1,周末/节假日代理,0,旺季,1,0,2.0,0,1,1,0,0,181+,0,Transient|Direct|Direct|0,0.0,1,9199,314.418213,15,高价值,1,5280,0.145076,26249,0.567107,8942,0.247372,1354,0.242245,1,0,82.035667,96.760925,107.124695,0.000000,0.000000,0.000000,0.628396,1,6,0.352941,1,0.468359
1,Resort Hotel - Mumbai,0,737,2024,April,17,28,0,0,2,0.0,0,BB,PRT,Direct,Direct,0,0,0,C,C,4,No Deposit,0.0,0.0,0,Transient,0.0,0,0,Check-Out,2024-04-28 21:56:21.507509066,Mumbai,0,2024-04-28,Resort Hotel,4,2024-04,2,2024Q2,6,1,1,周末/节假日代理,0,平季,0,0,2.0,0,1,1,0,0,181+,0,Transient|Direct|Direct|0,0.0,1,9199,314.418213,15,高价值,1,3007,0.143665,14885,0.565267,5052,0.251188,754,0.263926,0,0,83.775848,96.044243,106.139435,0.000000,0.000000,0.000000,0.565583,1,6,0.375000,1,0.473831
2,Resort Hotel - Delhi,0,7,2024,September,37,10,0,1,1,0.0,0,BB,GBR,Direct,Direct,0,0,0,A,C,0,No Deposit,0.0,0.0,0,Transient,75.0,0,0,Check-Out,2024-09-10 03:46:25.734029096,Delhi,1,2024-09-10,Resort Hotel,9,2024-09,3,2024Q3,1,0,0,工作日,0,平季,0,0,1.0,0,0,0,0,0,0-7,0,Transient|Direct|Direct|0,75.0,1,9199,314.418213,15,高价值,1,6356,0.147892,7365,0.210319,10815,0.249376,1571,0.250159,0,0,81.355942,96.428558,107.140785,0.921875,0.777778,0.700014,0.561910,1,8,0.444444,2,0.278788
3,Resort Hotel - Kolkata,0,13,2024,August,33,14,0,1,1,0.0,0,BB,GBR,Corporate,Corporate,0,0,0,A,A,0,No Deposit,304.0,0.0,0,Transient,75.0,0,0,Check-Out,2024-08-14 18:07:10.049669568,Kolkata,1,2024-08-14,Resort Hotel,8,2024-08,3,2024Q3,2,0,0,工作日,0,平季,0,0,1.0,0,1,0,0,0,8-30,0,Transient|Corporate|Corporate|0,75.0,1,2036,123.485298,13,高价值,1,1273,0.227808,6635,0.211756,0,0.000000,1392,0.274425,0,0,85.713806,97.885902,67.879440,0.875005,0.766198,1.104900,0.560142,4,12,0.857143,3,0.273685
4,Resort Hotel - Lucknow,0,14,2024,September,37,14,0,2,2,0.0,0,BB,GBR,Online TA,TA/TO,0,0,0,A,A,0,No Deposit,240.0,0.0,0,Transient,98.0,0,1,Check-Out,2024-09-14 14:27:32.473846000,Lucknow,2,2024-09-14,Resort Hotel,9,2024-09,3,2024Q3,5,1,1,周末/节假日代理,0,平季,0,0,2.0,0,1,0,1,0,8-30,1,Transient|Online TA|TA/TO|0,196.0,1,45891,368.200653,15,高价值,1,32492,0.379416,7493,0.210863,8065,0.364414,1521,0.249836,0,0,83.541191,96.566704,112.119072,1.173074,1.014843,0.874071,0.566750,2,

## 3. 建模字段选择

先剔除明显不能作为预测输入的字段，例如目标变量、日期字段和预订最终状态字段。`reservation_status` 和 `reservation_status_date` 与取消结果高度相关，属于结果发生后的信息，不作为建模输入。

In [3]:
# 明确建模候选字段，剔除目标变量、结果字段和日期原值。
drop_columns = [
    TARGET_COLUMN,
    "reservation_status",
    "reservation_status_date",
    "arrival_date",
]
existing_drop_columns = [col for col in drop_columns if col in df.columns]

candidate_features = df.drop(columns=existing_drop_columns).copy()
y = df[TARGET_COLUMN].astype("int8")

datetime_columns = candidate_features.select_dtypes(include=["datetime"]).columns.tolist()
if datetime_columns:
    candidate_features = candidate_features.drop(columns=datetime_columns)

print(f"候选特征数量：{candidate_features.shape[1]}")
print(f"剔除字段：{existing_drop_columns + datetime_columns}")
candidate_features.dtypes.value_counts()

候选特征数量：82
剔除字段：['is_canceled', 'reservation_status', 'reservation_status_date', 'arrival_date']


int8        31
float32     19
str          8
int16        4
int32        4
int64        3
category     1
category     1
category     1
category     1
category     1
category     1
category     1
category     1
category     1
category     1
category     1
category     1
float64      1
Name: count, dtype: int64

## 4. 划分训练集、验证集与测试集

后续所有需要学习参数的预处理步骤都只在训练集上拟合。

In [4]:
# 先按 7/2/1 分层划分数据，保证三份数据的取消率接近。
X_train_raw, X_temp_raw, y_train, y_temp = train_test_split(
    candidate_features,
    y,
    test_size=0.3,
    random_state=RANDOM_STATE,
    stratify=y,
)

X_validation_raw, X_test_raw, y_validation, y_test = train_test_split(
    X_temp_raw,
    y_temp,
    test_size=1 / 3,
    random_state=RANDOM_STATE,
    stratify=y_temp,
)

print(f"训练集：{X_train_raw.shape[0]:,} 行，取消率 {y_train.mean():.2%}")
print(f"验证集：{X_validation_raw.shape[0]:,} 行，取消率 {y_validation.mean():.2%}")
print(f"测试集：{X_test_raw.shape[0]:,} 行，取消率 {y_test.mean():.2%}")

训练集：77,439 行，取消率 37.08%
验证集：22,126 行，取消率 37.08%
测试集：11,063 行，取消率 37.08%


## 5. 类别特征识别与编码策略

高维类别字段使用 Target Encoding；低维类别字段使用 One-Hot Encoding。

In [5]:
# 识别类别字段，并区分高维类别字段和低维类别字段。
categorical_columns = X_train_raw.select_dtypes(include=["object", "category"]).columns.tolist()
category_cardinality = (
    X_train_raw[categorical_columns]
    .nunique(dropna=False)
    .sort_values(ascending=False)
    .rename("n_unique")
    .reset_index()
    .rename(columns={"index": "feature_name"})
)

# 根据 instruction，将国家和代理商作为重点高维 Target Encoding 字段；
# 同时对 hotel、customer_group_key 这类高基数字段也采用 Target Encoding，避免 One-Hot 维度过高。
target_encoding_columns = [
    col
    for col in ["country", "agent", "hotel", "customer_group_key"]
    if col in X_train_raw.columns
]
low_cardinality_categorical_columns = [
    col for col in categorical_columns if col not in target_encoding_columns
]

category_cardinality

C:\Users\Ted\AppData\Local\Temp\ipykernel_10148\3359451858.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_columns = X_train_raw.select_dtypes(include=["object", "category"]).columns.tolist()


,feature_name,n_unique
0,country,166
1,customer_group_key,109
2,hotel,30
3,city,15
4,arrival_month,12
5,arrival_date_month,12
6,assigned_room_type,12
7,reserved_room_type,10
8,market_segment,8
9,special_request_group,6


In [6]:
# 定义编码辅助函数，并只在训练集上拟合 Target Encoding 映射。
def sanitize_column_name(name):
    """清理编码后的列名，避免特殊字符影响后续保存和建模。"""
    return re.sub(r"[^0-9a-zA-Z_\u4e00-\u9fff]+", "_", str(name)).strip("_")


def fit_target_encoding(train_frame, target, columns, smoothing=20):
    """在训练集上拟合平滑 Target Encoding 映射。"""
    global_mean = float(target.mean())
    mappings = {}
    train_target = train_frame.copy()
    train_target[TARGET_COLUMN] = target.values

    for column in columns:
        stats = train_target.groupby(column, dropna=False, observed=False)[TARGET_COLUMN].agg(["mean", "count"])
        smooth_value = (stats["mean"] * stats["count"] + global_mean * smoothing) / (stats["count"] + smoothing)
        mappings[column] = smooth_value.to_dict()

    return mappings, global_mean


def transform_target_encoding(frame, columns, mappings, global_mean):
    """使用训练集映射转换类别字段，未知类别填充训练集目标均值。"""
    encoded = pd.DataFrame(index=frame.index)
    for column in columns:
        encoded[f"{column}_target_encoded"] = (
            frame[column].map(mappings[column]).fillna(global_mean).astype("float32")
        )
    return encoded


target_encoding_mappings, train_global_cancel_rate = fit_target_encoding(
    X_train_raw,
    y_train,
    target_encoding_columns,
)
target_encoded_train = transform_target_encoding(
    X_train_raw,
    target_encoding_columns,
    target_encoding_mappings,
    train_global_cancel_rate,
)
target_encoded_validation = transform_target_encoding(
    X_validation_raw,
    target_encoding_columns,
    target_encoding_mappings,
    train_global_cancel_rate,
)
target_encoded_test = transform_target_encoding(
    X_test_raw,
    target_encoding_columns,
    target_encoding_mappings,
    train_global_cancel_rate,
)

target_encoded_train.head()

,country_target_encoded,agent_target_encoded,hotel_target_encoded,customer_group_key_target_encoded
69482,0.567504,0.736224,0.419418,0.979480
107456,0.240559,0.159395,0.413146,0.448465
79832,0.243253,0.122470,0.413146,0.379617
5577,0.240559,0.582509,0.270562,0.979480
108343,0.169598,0.122470,0.408515,0.379617


In [7]:
# 对低维类别字段做 One-Hot，并以训练集列为基准对齐验证集和测试集。
one_hot_train = pd.get_dummies(
    X_train_raw[low_cardinality_categorical_columns].astype(str),
    prefix=[sanitize_column_name(col) for col in low_cardinality_categorical_columns],
    dummy_na=False,
    dtype="int8",
)
one_hot_validation = pd.get_dummies(
    X_validation_raw[low_cardinality_categorical_columns].astype(str),
    prefix=[sanitize_column_name(col) for col in low_cardinality_categorical_columns],
    dummy_na=False,
    dtype="int8",
)
one_hot_test = pd.get_dummies(
    X_test_raw[low_cardinality_categorical_columns].astype(str),
    prefix=[sanitize_column_name(col) for col in low_cardinality_categorical_columns],
    dummy_na=False,
    dtype="int8",
)
one_hot_train, one_hot_validation = one_hot_train.align(one_hot_validation, join="left", axis=1, fill_value=0)
one_hot_train, one_hot_test = one_hot_train.align(one_hot_test, join="left", axis=1, fill_value=0)
one_hot_validation = one_hot_validation.reindex(columns=one_hot_train.columns, fill_value=0).astype("int8")
one_hot_test = one_hot_test.reindex(columns=one_hot_train.columns, fill_value=0).astype("int8")

print(f"Target Encoding 字段：{target_encoding_columns}")
print(f"One-Hot 字段数：{len(low_cardinality_categorical_columns)}")
print(f"One-Hot 后维度：{one_hot_train.shape[1]}")

Target Encoding 字段：['country', 'agent', 'hotel', 'customer_group_key']
One-Hot 字段数：17
One-Hot 后维度：112


## 6. 数值型特征标准化

In [8]:
# 对数值字段做缺失检查，并使用训练集拟合 StandardScaler。
numeric_columns = X_train_raw.select_dtypes(include=["number", "bool"]).columns.tolist()
numeric_columns = [col for col in numeric_columns if col not in target_encoding_columns]

numeric_train = X_train_raw[numeric_columns].copy()
numeric_validation = X_validation_raw[numeric_columns].copy()
numeric_test = X_test_raw[numeric_columns].copy()

numeric_missing_summary = pd.DataFrame(
    {
        "train_missing": numeric_train.isna().sum(),
        "validation_missing": numeric_validation.isna().sum(),
        "test_missing": numeric_test.isna().sum(),
    }
)
numeric_missing_summary = numeric_missing_summary[
    (numeric_missing_summary["train_missing"] > 0)
    | (numeric_missing_summary["validation_missing"] > 0)
    | (numeric_missing_summary["test_missing"] > 0)
]
if not numeric_missing_summary.empty:
    raise ValueError("数值字段仍存在缺失值，请先回到数据清洗或特征工程阶段处理。")

scaler = StandardScaler()
scaled_train = pd.DataFrame(
    scaler.fit_transform(numeric_train),
    columns=[f"{col}_scaled" for col in numeric_columns],
    index=X_train_raw.index,
)
scaled_validation = pd.DataFrame(
    scaler.transform(numeric_validation),
    columns=[f"{col}_scaled" for col in numeric_columns],
    index=X_validation_raw.index,
)
scaled_test = pd.DataFrame(
    scaler.transform(numeric_test),
    columns=[f"{col}_scaled" for col in numeric_columns],
    index=X_test_raw.index,
)

print(f"标准化数值特征数量：{len(numeric_columns)}")
scaled_train.describe().T.head(65)

标准化数值特征数量：61


,count,mean,std,min,25%,50%,75%,max
lead_time_scaled,77439.0,1.981911e-17,1.000006,-0.957473,-0.800340,-0.328942,0.512182,5.854701
arrival_date_year_scaled,77439.0,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
arrival_date_week_number_scaled,77439.0,2.752655e-19,1.000006,-1.688744,-0.889208,-0.023045,0.843119,1.709282
arrival_date_day_of_month_scaled,77439.0,3.761961e-17,1.000006,-1.672364,-0.877379,0.031175,0.826159,1.734713
stays_in_weekend_nights_scaled,77439.0,3.927121e-17,1.000006,-0.978465,-0.978465,0.219425,1.417316,1.417316
...,...,...,...,...,...,...,...,...
price_sensitivity_score_scaled,77439.0,-3.156377e-17,1.000006,-1.416671,-0.709800,-0.002930,0.703940,1.410810
hotel_daily_booking_count_scaled,77439.0,1.614891e-17,1.000006,-2.344649,-0.733380,0.072255,0.676481,3.496202
estimated_occupancy_proxy_scaled,77439.0,3.015992e-16,1.000006,-3.127182,-0.717192,-0.035634,0.680002,2.826910
demand_pressure_score_scaled,77439.0,-8.670862e-17,1.000006,-1.438202,-0.703256,0.031689,0.766635,1.501581


## 7. 合并预处理后特征

In [9]:
# 合并标准化数值特征、Target Encoding 特征和 One-Hot 特征。
X_train_processed = pd.concat(
    [scaled_train, target_encoded_train, one_hot_train],
    axis=1,
)
X_validation_processed = pd.concat(
    [scaled_validation, target_encoded_validation, one_hot_validation],
    axis=1,
)
X_test_processed = pd.concat(
    [scaled_test, target_encoded_test, one_hot_test],
    axis=1,
)

X_train_processed.columns = [sanitize_column_name(col) for col in X_train_processed.columns]
X_validation_processed.columns = [sanitize_column_name(col) for col in X_validation_processed.columns]
X_test_processed.columns = [sanitize_column_name(col) for col in X_test_processed.columns]

print(f"预处理后训练特征维度：{X_train_processed.shape}")
print(f"预处理后验证特征维度：{X_validation_processed.shape}")
print(f"预处理后测试特征维度：{X_test_processed.shape}")
X_train_processed.head()

预处理后训练特征维度：(77439, 177)
预处理后验证特征维度：(22126, 177)
预处理后测试特征维度：(11063, 177)


,lead_time_scaled,arrival_date_year_scaled,arrival_date_week_number_scaled,arrival_date_day_of_month_scaled,stays_in_weekend_nights_scaled,stays_in_week_nights_scaled,adults_scaled,children_scaled,babies_scaled,is_repeated_guest_scaled,previous_cancellations_scaled,previous_bookings_not_canceled_scaled,booking_changes_scaled,company_scaled,days_in_waiting_list_scaled,adr_scaled,required_car_parking_spaces_scaled,total_of_special_requests_scaled,total_stays_scaled,arrival_month_num_scaled,arrival_quarter_scaled,arrival_day_of_week_scaled,is_weekend_scaled,is_holiday_proxy_scaled,is_near_holiday_proxy_scaled,is_peak_season_scaled,is_low_season_scaled,total_guests_scaled,has_children_or_babies_scaled,room_type_matched_scaled,has_booking_changes_scaled,has_special_requests_scaled,has_car_parking_request_scaled,estimated_revenue_scaled,group_recency_days_scaled,group_booking_count_scaled,group_avg_revenue_scaled,rfm_value_score_scaled,is_high_value_customer_group_scaled,customer_group_hist_booking_count_scaled,customer_group_hist_cancel_rate_scaled,country_hist_booking_count_scaled,country_hist_cancel_rate_scaled,agent_hist_booking_count_scaled,agent_hist_cancel_rate_scaled,hotel_hist_booking_count_scaled,hotel_hist_cancel_rate_scaled,is_high_risk_country_scaled,is_high_risk_agent_scaled,hotel_hist_avg_adr_scaled,city_hist_avg_adr_scaled,customer_group_hist_avg_adr_scaled,price_to_hotel_hist_avg_scaled,price_to_city_hist_avg_scaled,price_to_customer_group_hist_avg_scaled,price_volatility_index_scaled,price_sensitivity_score_scaled,hotel_daily_booking_count_scaled,estimated_occupancy_proxy_scaled,demand_pressure_score_scaled,...,deposit_type_No_Deposit,deposit_type_Non_Refund,deposit_type_Refundable,customer_type_Contract,customer_type_Group,customer_type_Transient,customer_type_Transient_Party,city_Ahmedabad,city_Bangalore,city_Bhopal,city_Chandigarh,city_Chennai,city_Delhi,city_Goa,city_Hyderabad,city_Indore,city_Jaipur,city_Kochi,city_Kolkata,city_Lucknow,city_Mumbai,city_Pune,hotel_type_City_Hotel,hotel_type_Resort_Hotel,arrival_month_2024_01,arrival_month_2024_02,arrival_month_2024_03,arrival_month_2024_04,arrival_month_2024_05,arrival_month_2024_06,arrival_month_2024_07,arrival_month_2024_08,arrival_month_2024_09,arrival_month_2024_10,arrival_month_2024_11,arrival_month_2024_12,arrival_quarter_label_2024Q1,arrival_quarter_label_2024Q2,arrival_quarter_label_2024Q3,arrival_quarter_label_2024Q4,holiday_proxy_周末_节假日代理,holiday_proxy_工作日,season_type_平季,season_type_旺季,season_type_淡季,lead_time_group_0_7,lead_time_group_181,lead_time_group_31_90,lead_time_group_8_30,lead_time_group_91_180,special_request_group_0,special_request_group_1,special_request_group_2,special_request_group_3,special_request_group_4,special_request_group_5,rfm_segment_沉睡,rfm_segment_流失,rfm_segment_潜力,rfm_segment_高价值
69482,2.203671,0.0,-0.289557,1.280436,1.417316,1.307726,0.27504,-0.234313,-0.085284,-0.186093,1.051894,-0.094733,-0.339187,-0.204301,-0.136109,1.801249,-0.247677,-0.710033,1.723022,-0.430202,-0.444048,-1.499243,-0.634749,-0.634749,1.585674,-0.590041,-0.564311,0.098751,-0.263257,0.387679,-0.413514,-0.819848,-0.251041,3.166488,-0.057284,-0.812407,-0.760312,0.36833,0.120647,-0.660026,2.913634,0.550186,1.086454,-0.437883,1.838206,-0.042121,0.651982,-0.562223,1.797676,0.701980,0.300923,-0.515924,1.505748,1.779927,1.611955,-0.612821,1.410810,-0.129154,-0.512725,0.031689,...,0,1,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,1,1,0,0,0,1,0,0,0,1,0,0,0,0,0,0,0,0,1
107456,-0.560019,0.0,0.110211,-0.423102,-0.978465,-0.907692,0.27504,-0.234313,-0.085284,-0.186093,-0.106557,-0.094733,-0.339187,-0.204301,-0.136109,-0.643445,-0.247677,-0.710033,-1.193372,0.152269,0.452348,0.498888,-0.634749,-0.634749,1.585674,1.694797,-0.564311,0.098751,-0.263257,0.387679,-0.413514,-0.819848,-0.251041,-1.024990,-0.057284,-0.536222,-0.015428,0.36833,0.120647,-0.359031,0.361868,-0.524479,-0.659358,-0.796458,-0.932825,0.471197,0.782950,-0.562223,-0.55627

## 8. 方差阈值法特征筛选

In [10]:
# 使用方差阈值删除常量特征，避免无信息字段进入后续评估。
variance_selector = VarianceThreshold(threshold=0.0)
variance_selector.fit(X_train_processed)

variance_report = pd.DataFrame(
    {
        "feature_name": X_train_processed.columns,
        "variance": variance_selector.variances_,
        "kept_by_variance_threshold": variance_selector.get_support(),
    }
)

kept_features_by_variance = variance_report.loc[
    variance_report["kept_by_variance_threshold"], "feature_name"
].tolist()
X_train_variance = X_train_processed[kept_features_by_variance]
X_validation_variance = X_validation_processed[kept_features_by_variance]
X_test_variance = X_test_processed[kept_features_by_variance]

print(f"方差阈值删除特征数：{len(X_train_processed.columns) - len(kept_features_by_variance)}")
variance_report.sort_values("variance").head(20)

方差阈值删除特征数：1


,feature_name,variance,kept_by_variance_threshold
1,arrival_date_year_scaled,0.000000,False
115,assigned_room_type_L,0.000013,True
89,market_segment_Undefined,0.000026,True
94,distribution_channel_Undefined,0.000065,True
103,reserved_room_type_L,0.000077,True
104,reserved_room_type_P,0.000129,True
116,assigned_room_type_P,0.000129,True
172,special_request_group_5,0.000258,True
174,rfm_segment_流失,0.000297,True
119,deposit_type_Refundable,0.001393,True


## 9. 相关性分析

In [11]:
# 计算每个特征与目标变量的线性相关性，作为特征有效性的初步参考。
correlation_with_target = X_train_variance.corrwith(y_train).fillna(0)
correlation_report = (
    correlation_with_target.rename("target_correlation")
    .reset_index()
    .rename(columns={"index": "feature_name"})
)
correlation_report["abs_target_correlation"] = correlation_report["target_correlation"].abs()
correlation_report = correlation_report.sort_values("abs_target_correlation", ascending=False)
correlation_report.head(200)

,feature_name,target_correlation,abs_target_correlation
59,cancellation_risk_score_scaled,0.577929,0.577929
117,deposit_type_Non_Refund,0.500491,0.500491
116,deposit_type_No_Deposit,-0.496744,0.496744
63,customer_group_key_target_encoded,0.429232,0.429232
39,customer_group_hist_cancel_rate_scaled,0.427846,0.427846
...,...,...,...
152,arrival_quarter_label_2024Q1,0.000246,0.000246
134,city_Kolkata,-0.000140,0.000140
129,city_Goa,-0.000087,0.000087
67,arrival_date_month_February,-0.000077,0.000077


## 10. 互信息法特征筛选

In [12]:
# 使用互信息评估非线性关系，抽样计算是为了控制运行时间。
mi_sample_size = min(30000, X_train_variance.shape[0])
mi_sample = X_train_variance.sample(n=mi_sample_size, random_state=RANDOM_STATE)
y_mi_sample = y_train.loc[mi_sample.index]

mutual_information = mutual_info_classif(
    mi_sample,
    y_mi_sample,
    discrete_features=False,
    random_state=RANDOM_STATE,
)
mi_report = pd.DataFrame(
    {
        "feature_name": mi_sample.columns,
        "mutual_information": mutual_information,
    }
).sort_values("mutual_information", ascending=False)

mi_report.head(20)

,feature_name,mutual_information
59,cancellation_risk_score_scaled,0.209631
117,deposit_type_Non_Refund,0.144883
116,deposit_type_No_Deposit,0.142670
35,group_avg_revenue_scaled,0.104549
34,group_booking_count_scaled,0.104209
63,customer_group_key_target_encoded,0.102418
39,customer_group_hist_cancel_rate_scaled,0.101063
32,estimated_revenue_scaled,0.091749
0,lead_time_scaled,0.091216
50,customer_group_hist_avg_adr_scaled,0.083134


## 11. 特征重要性预评估

这里使用随机森林做预评估，目的不是最终建模，而是快速观察特征有效性。

In [13]:
# 使用随机森林做特征重要性预评估，此处不作为最终模型结论。
rf_model = RandomForestClassifier(
    n_estimators=120,
    max_depth=8,
    min_samples_leaf=50,
    n_jobs=-1,
    random_state=RANDOM_STATE,
    class_weight="balanced_subsample",
)
rf_model.fit(X_train_variance, y_train)

train_auc = roc_auc_score(y_train, rf_model.predict_proba(X_train_variance)[:, 1])
validation_auc = roc_auc_score(y_validation, rf_model.predict_proba(X_validation_variance)[:, 1])
test_auc = roc_auc_score(y_test, rf_model.predict_proba(X_test_variance)[:, 1])

feature_importance_report = pd.DataFrame(
    {
        "feature_name": X_train_variance.columns,
        "rf_importance": rf_model.feature_importances_,
    }
).sort_values("rf_importance", ascending=False)

print(f"随机森林预评估训练 AUC：{train_auc:.4f}")
print(f"随机森林预评估验证 AUC：{validation_auc:.4f}")
print(f"随机森林预评估测试 AUC：{test_auc:.4f}")
feature_importance_report.head(30)

随机森林预评估训练 AUC：0.9307
随机森林预评估验证 AUC：0.9284
随机森林预评估测试 AUC：0.9277


,feature_name,rf_importance
59,cancellation_risk_score_scaled,0.156414
117,deposit_type_Non_Refund,0.071909
116,deposit_type_No_Deposit,0.070196
60,country_target_encoded,0.057665
41,country_hist_cancel_rate_scaled,0.051328
61,agent_target_encoded,0.049394
43,agent_hist_cancel_rate_scaled,0.044156
63,customer_group_key_target_encoded,0.043059
0,lead_time_scaled,0.032954
28,room_type_matched_scaled,0.032374


## 12. 汇总特征有效性报告

In [14]:
# 汇总方差、相关性、互信息和随机森林重要性，形成特征预评估报告。
feature_selection_report = (
    variance_report
    .merge(correlation_report, on="feature_name", how="left")
    .merge(mi_report, on="feature_name", how="left")
    .merge(feature_importance_report, on="feature_name", how="left")
)
feature_selection_report[["target_correlation", "abs_target_correlation", "mutual_information", "rf_importance"]] = (
    feature_selection_report[["target_correlation", "abs_target_correlation", "mutual_information", "rf_importance"]]
    .fillna(0)
)

feature_selection_report["has_target_encoded_source"] = feature_selection_report["feature_name"].str.contains(
    "target_encoded", regex=False
)
feature_selection_report["is_target_derived_feature"] = feature_selection_report["feature_name"].str.contains(
    "hist_cancel_rate|risk_score|high_risk", regex=True
)
feature_selection_report["overall_preview_rank"] = (
    feature_selection_report["abs_target_correlation"].rank(ascending=False)
    + feature_selection_report["mutual_information"].rank(ascending=False)
    + feature_selection_report["rf_importance"].rank(ascending=False)
)
feature_selection_report = feature_selection_report.sort_values("overall_preview_rank")

feature_selection_report.head(30)

,feature_name,variance,kept_by_variance_threshold,target_correlation,abs_target_correlation,mutual_information,rf_importance,has_target_encoded_source,is_target_derived_feature,overall_preview_rank
60,cancellation_risk_score_scaled,1.000000,True,0.577929,0.577929,0.209631,0.156414,False,True,3.0
118,deposit_type_Non_Refund,0.113796,True,0.500491,0.500491,0.144883,0.071909,False,False,6.0
117,deposit_type_No_Deposit,0.114823,True,-0.496744,0.496744,0.142670,0.070196,False,False,9.0
64,customer_group_key_target_encoded,0.041320,True,0.429232,0.429232,0.102418,0.043059,True,False,18.0
40,customer_group_hist_cancel_rate_scaled,1.000000,True,0.427846,0.427846,0.101063,0.030766,False,True,23.0
62,agent_target_encoded,0.030879,True,0.389373,0.389373,0.078465,0.049394,True,False,24.0
61,country_target_encoded,0.029661,True,0.361110,0.361110,0.070054,0.057665,True,False,26.0
44,agent_hist_cancel_rate_scaled,1.000000,True,0.383608,0.383608,0.078309,0.044156,False,True,27.0
0,lead_time_scaled,1.000000,True,0.310474,0.310474,0.091216,0.032954,False,False,28.0
42,country_hist_cancel_rate_scaled,1.000000,True,0.355326,0.355326,0.066568,0.051328,False,True,29.0


## 13. 保存建模数据和报告

In [15]:
# 保存训练、验证、测试三份建模数据和对应的特征评估报告。
train_output = X_train_variance.copy()
train_output[TARGET_COLUMN] = y_train.values
train_output["dataset_split"] = "train"

validation_output = X_validation_variance.copy()
validation_output[TARGET_COLUMN] = y_validation.values
validation_output["dataset_split"] = "validation"

test_output = X_test_variance.copy()
test_output[TARGET_COLUMN] = y_test.values
test_output["dataset_split"] = "test"

modeling_dataset = pd.concat([train_output, validation_output, test_output], axis=0).sort_index()

MODELING_DATASET_PATH.parent.mkdir(parents=True, exist_ok=True)
PREPROCESSING_SUMMARY_PATH.parent.mkdir(parents=True, exist_ok=True)

train_output.to_parquet(TRAIN_DATASET_PATH, index=False)
validation_output.to_parquet(VALIDATION_DATASET_PATH, index=False)
test_output.to_parquet(TEST_DATASET_PATH, index=False)
modeling_dataset.to_parquet(MODELING_DATASET_PATH, index=False)
feature_selection_report.to_csv(FEATURE_SELECTION_REPORT_PATH, index=False, encoding="utf-8-sig")
feature_importance_report.to_csv(FEATURE_IMPORTANCE_REPORT_PATH, index=False, encoding="utf-8-sig")

preprocessing_summary = pd.DataFrame(
    [
        ("input_rows", df.shape[0]),
        ("input_columns", df.shape[1]),
        ("candidate_features", candidate_features.shape[1]),
        ("train_rows", X_train_raw.shape[0]),
        ("validation_rows", X_validation_raw.shape[0]),
        ("test_rows", X_test_raw.shape[0]),
        ("numeric_scaled_features", len(numeric_columns)),
        ("target_encoded_features", len(target_encoding_columns)),
        ("one_hot_source_features", len(low_cardinality_categorical_columns)),
        ("one_hot_output_features", one_hot_train.shape[1]),
        ("processed_train_features_before_variance", X_train_processed.shape[1]),
        ("processed_train_features_after_variance", X_train_variance.shape[1]),
        ("random_forest_preview_train_auc", round(train_auc, 6)),
        ("random_forest_preview_validation_auc", round(validation_auc, 6)),
        ("random_forest_preview_test_auc", round(test_auc, 6)),
    ],
    columns=["metric", "value"],
)
preprocessing_summary.to_csv(PREPROCESSING_SUMMARY_PATH, index=False, encoding="utf-8-sig")

print(f"训练集建模数据：{TRAIN_DATASET_PATH.relative_to(PROJECT_ROOT)}")
print(f"验证集建模数据：{VALIDATION_DATASET_PATH.relative_to(PROJECT_ROOT)}")
print(f"测试集建模数据：{TEST_DATASET_PATH.relative_to(PROJECT_ROOT)}")
print(f"合并建模数据：{MODELING_DATASET_PATH.relative_to(PROJECT_ROOT)}")
print(f"特征筛选报告：{FEATURE_SELECTION_REPORT_PATH.relative_to(PROJECT_ROOT)}")
print(f"特征重要性预评估报告：{FEATURE_IMPORTANCE_REPORT_PATH.relative_to(PROJECT_ROOT)}")
print(f"预处理摘要：{PREPROCESSING_SUMMARY_PATH.relative_to(PROJECT_ROOT)}")

preprocessing_summary

训练集建模数据：data\processed\modeling_dataset_train.parquet
验证集建模数据：data\processed\modeling_dataset_validation.parquet
测试集建模数据：data\processed\modeling_dataset_test.parquet
合并建模数据：data\processed\modeling_dataset.parquet
特征筛选报告：reports\feature_selection_report.csv
特征重要性预评估报告：reports\feature_importance_preview.csv
预处理摘要：reports\feature_preprocessing_summary.csv


,metric,value
0,input_rows,110628.000000
1,input_columns,86.000000
2,candidate_features,82.000000
3,train_rows,77439.000000
4,validation_rows,22126.000000
5,test_rows,11063.000000
6,numeric_scaled_features,61.000000
7,target_encoded_features,4.000000
8,one_hot_source_features,17.000000
9,one_hot_output_features,112.000000


## 14. 阶段性结论

In [16]:
# 输出本阶段总结，强调编码和标准化都只在训练集上学习规则。
summary = [
    "已完成训练/验证/测试集 7/2/1 分层划分，三份数据的目标变量分布基本一致。",
    "数值型特征已使用 StandardScaler 标准化；标准化参数只在训练集拟合，再应用到验证集和测试集。",
    "当前数值型特征不存在缺失值，因此本阶段不再额外填补，若发现缺失将提示回到清洗或特征工程阶段处理。",
    "国家、代理商、酒店和客户群体等高维类别字段已使用平滑 Target Encoding 处理，映射关系只在训练集学习。",
    "低维类别字段已使用 One-Hot Encoding，并以训练集编码列为基准对齐验证集和测试集。",
    "已通过方差阈值、相关性分析、互信息法和随机森林重要性完成特征有效性预评估，最终保留 176 个建模特征。",
    "历史取消率、Target Encoding 和取消风险评分均与目标变量相关，后续正式建模时需要继续严格控制数据泄露。",
]

for item in summary:
    print("- " + item)


- 已完成训练/验证/测试集 7/2/1 分层划分，三份数据的目标变量分布基本一致。
- 数值型特征已使用 StandardScaler 标准化；标准化参数只在训练集拟合，再应用到验证集和测试集。
- 当前数值型特征不存在缺失值，因此本阶段不再额外填补，若发现缺失将提示回到清洗或特征工程阶段处理。
- 国家、代理商、酒店和客户群体等高维类别字段已使用平滑 Target Encoding 处理，映射关系只在训练集学习。
- 低维类别字段已使用 One-Hot Encoding，并以训练集编码列为基准对齐验证集和测试集。
- 已通过方差阈值、相关性分析、互信息法和随机森林重要性完成特征有效性预评估，最终保留 176 个建模特征。
- 历史取消率、Target Encoding 和取消风险评分均与目标变量相关，后续正式建模时需要继续严格控制数据泄露。
